In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import json
import pandas as pd
from tqdm import tqdm

In [ ]:
INPUT_DIR = "/content/drive/MyDrive/PK-3/data/raw"

OUTPUT_DIR = "/content/drive/MyDrive/PK-3/data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def extract_no_perkara(text):

    match = re.search(
        r"Nomor\s+([^\n]+)",
        text,
        re.IGNORECASE
    )

    return match.group(1).strip() if match else ""

In [ ]:
def extract_pihak(text):

    match = re.search(
        r"Nama lengkap\s*:?\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    if match:

        nama = match.group(1)

        nama = re.sub(
            r"\s+",
            " ",
            nama
        )

        nama = nama.replace(
            ";",
            ""
        )

        return nama.strip()

    return "Tidak ditemukan"

In [ ]:
def extract_tanggal(text):

    match = re.search(
        r"tanggal\s+(\d{1,2}\s+\w+\s+\d{4})",
        text,
        re.IGNORECASE
    )

    return match.group(1) if match else ""

In [ ]:
from datetime import datetime
import re

def extract_tanggal(text):

    bulan = {
        "januari": "01",
        "februari": "02",
        "maret": "03",
        "april": "04",
        "mei": "05",
        "juni": "06",
        "juli": "07",
        "agustus": "08",
        "september": "09",
        "oktober": "10",
        "november": "11",
        "desember": "12"
    }

    match = re.search(
        r"tanggal\s+(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})",
        text,
        re.IGNORECASE
    )

    if not match:
        return ""

    hari = match.group(1).zfill(2)
    nama_bulan = match.group(2).lower()
    tahun = match.group(3)

    if nama_bulan not in bulan:
        return ""

    return f"{tahun}-{bulan[nama_bulan]}-{hari}"

In [ ]:
def extract_pasal(text):

    pasal = re.findall(
        r"Pasal\s+\d+\s+KUHP",
        text,
        re.IGNORECASE
    )

    pasal = list(dict.fromkeys(pasal))

    return "; ".join(pasal)

In [ ]:
def extract_ringkasan_fakta(text):

    # Prioritas 1:
    # Ambil bagian antara DAKWAAN dan MENIMBANG

    match = re.search(
        r"DAKWAAN(.*?)(?=MENIMBANG|Menimbang|$)",
        text,
        re.IGNORECASE | re.DOTALL
    )

    if match:

        fakta = match.group(1)

        fakta = re.sub(
            r"\s+",
            " ",
            fakta
        ).strip()

        return fakta[:1000]

    # Prioritas 2:
    # Jika hanya menemukan DAKWAAN

    match = re.search(
        r"DAKWAAN(.*)",
        text,
        re.IGNORECASE | re.DOTALL
    )

    if match:

        fakta = re.sub(
            r"\s+",
            " ",
            match.group(1)
        ).strip()

        return fakta[:1000]

    # Prioritas 3:
    # Fallback agar tidak kosong / NaN

    fakta = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return fakta[:1000]

In [ ]:
def extract_amar_putusan(text):

    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""

    # Normalisasi newline
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # Hapus boilerplate MA yang sering muncul di tengah dokumen
    text = re.sub(
        r'pelaksanaan fungsi peradilan.*?Mahkamah Agung RI melalui\s*:',
        '',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Cari heading amar putusan
    heading_patterns = [
        r'M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?',
        r'\n\s*MENGADILI\s*:?',
        r'\n\s*M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\b'
    ]

    matches = []

    for pattern in heading_patterns:
        matches.extend(
            list(
                re.finditer(
                    pattern,
                    text,
                    flags=re.IGNORECASE
                )
            )
        )

    if not matches:
        return ""

    # Ambil kemunculan TERAKHIR
    # karena kata "mengadili perkara pidana" sering muncul di awal putusan
    start_match = max(matches, key=lambda x: x.start())

    amar_text = text[start_match.end():]

    # Cari akhir amar
    end_patterns = [
        r'\n\s*Demikian',
        r'\n\s*Hakim Ketua',
        r'\n\s*Hakim Anggota',
        r'\n\s*Panitera',
        r'\n\s*Ditetapkan',
    ]

    end_positions = []

    for pattern in end_patterns:
        m = re.search(pattern, amar_text, flags=re.IGNORECASE)
        if m:
            end_positions.append(m.start())

    if end_positions:
        amar_text = amar_text[:min(end_positions)]

    # Rapikan whitespace
    amar_text = re.sub(r'\n{3,}', '\n\n', amar_text)
    amar_text = re.sub(r'[ \t]+', ' ', amar_text)

    # Ambil hanya poin amar jika ada
    amar_items = re.findall(
        r'(?ms)^\s*\d+\.\s.*?(?=^\s*\d+\.\s|\Z)',
        amar_text
    )

    if amar_items:
        amar_text = "\n".join(
            item.strip()
            for item in amar_items
        )

    return amar_text.strip()

In [ ]:
records = []

files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.endswith(".txt")
])

for idx, file in enumerate(tqdm(files), start=1):

    path = os.path.join(INPUT_DIR, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    records.append({

        "case_id": idx,

        "no_perkara":
            extract_no_perkara(text),

        "tanggal":
            extract_tanggal(text),

        "ringkasan_fakta":
            extract_ringkasan_fakta(text),

        "pasal":
            extract_pasal(text),

        "pihak":
            extract_pihak(text),

        "amar_putusan":
            extract_amar_putusan(text),

        "text_full":
            text
    })

100%|██████████| 50/50 [00:00<00:00, 109.47it/s]


In [ ]:
df = pd.DataFrame(records)

csv_path = os.path.join(
    OUTPUT_DIR,
    "cases.csv"
)

json_path = os.path.join(
    OUTPUT_DIR,
    "cases.json"
)

df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

df.to_json(
    json_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print("CSV :", csv_path)
print("JSON:", json_path)

df.head()

CSV : /content/drive/MyDrive/PK-3/data/processed/cases.csv
JSON: /content/drive/MyDrive/PK-3/data/processed/cases.json


,case_id,no_perkara,tanggal,ringkasan_fakta,pasal,pihak,amar_putusan,text_full
0,1,1074/Pid.B/2025/PN Bdg,2025-09-19,Kesatu Penuntut Umum; 2. Menjatuhkan pidana te...,Pasal 378 KUHP; Pasal\n378 KUHP; Pasal\n372 KU...,DERIN HARISTYA UKITA Bin SUHENDAR HARIS,1. Menyatakan Terdakwa DERIN HARISTYA UKITA Bi...,P U T U S A N\nNomor 1074/Pid.B/2025/PN Bdg\n\...
1,2,1091/Pid.B/2025/PN Bdg,2025-11-17,alternatif Kedua: Pasal 374 KUHP jo 64 ayat (1...,Pasal 374 KUHP; Pasal\n378 KUHP; Pasal 378 KUH...,MUHAMAD ISSYAK HUSEN AMD BIN DJANDJAN,1. Menyatakan Terdakwa MUHAMAD ISSYAK HUSEN AM...,P U T U S A N\nNomor 1091/Pid.B/2025/PN Bdg\nD...
2,3,1099/Pid.B/2025/PN Bdg,2025-09-20,Kesatu. 2. Menjatuhkan pidana terhadap Terdakw...,pasal 378\nKUHP; pasal 378 KUHP; pasal 372 KUH...,Ardy Putra Auwy anak dari Ferdianto Auwy,1. Menyatakan Terdakwa Ardy Putra Auwy anak da...,Hal. 1 dari 56 halaman Putusan Nomor 1099/Pid....
3,4,1100/Pid.B/2024/PN Bdg,2024-10-21,Alternatif Pertama; 2. Menjatuhkan pidana terh...,pasal 378 KUHP; Pasal 372\nKUHP; Pasal 378 KUHP,MUHAMMAD IKHSAN GUNAWAN Bin,1. Menyatakan terdakwa MUHAMMAD IKHSAN GUNAWAN...,P U T U S A N\nNomor 1100/Pid.B/2024/PN Bdg\nD...
4,5,1104/Pid.B/2025/PN Bdg,2025-02-25,alternatif Kesatu kami melanggar pasal 378 KUH...,pasal 378 KUHP; Pasal\n378 KUHP; pasal 372 KUH...,RUDY SYARIF,1.\nMenyatakan Terdakwa RUDY SYARIF telah terb...,P U T U S A N\nNomor 1104/Pid.B/2025/PN Bdg\n\...


In [ ]:
df["jenis_perkara"] = "Pidana Penipuan"

In [ ]:
df["word_count"] = (
    df["text_full"]
    .str.split()
    .str.len()
)

In [ ]:
df[["case_id", "word_count"]]

,case_id,word_count
0,1,11015
1,2,15822
2,3,22423
3,4,5466
4,5,76660
5,6,110394
6,7,90920
7,8,6847
8,9,2774
9,10,8696


In [ ]:
print(df.isna().sum())

case_id            0
no_perkara         0
tanggal            0
ringkasan_fakta    0
pasal              0
pihak              0
amar_putusan       0
text_full          0
dtype: int64
